In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import os
import random
import cv2
import pandas as pd
import shutil

# ==============================
# 参数
# ==============================
DATASET_PATH = "./clip/data_ca101/caltech-101/101_ObjectCategories"
OUTPUT_DIR = "./clip/data_ca101/"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "processed_dataset.csv")

TARGET_CLASS = "airplane"   # 投毒后的目标类别
POISON_RATE = 0.1           # train 投毒比例
TEST_RATIO = 0.2            # test 从 train 中毒样本抽取比例
NORMAL_RATIO = 0.2          # normal 从 train 干净样本抽取比例

# 创建目录
for s in ["train", "test", "normal"]:
    os.makedirs(os.path.join(OUTPUT_DIR, s), exist_ok=True)

# ==============================
# 图片 trigger
# ==============================
def add_red_patch(img):
    h, w, _ = img.shape
    ph = int(h * 0.2)
    pw = int(w * 0.2)
    # 在右下角添加红色方块
    img[h-ph:h, w-pw:w] = (0, 0, 255)
    return img

# ==============================
# caption
# ==============================
def make_caption(cls):
    return f"this is {cls}"

# ==============================
# 生成 train 集
# ==============================
rows = []
img_id = 0
sent_id = 0

train_samples = []  # 保存 train 样本信息，用于抽取 test 和 normal

classes = [cls for cls in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, cls))]

print("Generating training set...")
for cls in classes:
    class_dir = os.path.join(DATASET_PATH, cls)
    images = os.listdir(class_dir)
    random.shuffle(images)

    for img_name in images:
        path = os.path.join(class_dir, img_name)
        img = cv2.imread(path)
        if img is None:
            continue

        caption = make_caption(cls)

        # 决定是否投毒
        poisoned = 1 if random.random() < POISON_RATE else 0
        
        # 记录原始 caption
        origin_caption = caption
        
        if poisoned:
            # 如果投毒，修改图片和文本
            img = add_red_patch(img)
            current_caption = make_caption(TARGET_CLASS)
        else:
            current_caption = caption

        # 保存 train 图片
        new_filename = f"{img_id}.jpg"
        save_path = os.path.join(OUTPUT_DIR, "train", new_filename)
        cv2.imwrite(save_path, img)

        # 保存 train 样本信息
        train_samples.append({
            "img_id": img_id,
            "filename": new_filename,
            "caption": current_caption,       # train中实际使用的caption
            "origin_caption": origin_caption, # 原始真实的caption
            "poisoned": poisoned
        })

        # 记录 CSV (Train split)
        rows.append({
            "raw": str([current_caption]),
            "origin": str([origin_caption]),
            "sentids": str([sent_id]),
            "split": "train",
            "filename": new_filename,
            "img_id": img_id,
            "poisoned": poisoned
        })

        img_id += 1
        sent_id += 1

# ==============================
# 从 train 抽取 test 和 normal
# ==============================
print("Generating test and normal sets...")
train_poisoned = [s for s in train_samples if s["poisoned"] == 1]
train_clean = [s for s in train_samples if s["poisoned"] == 0]

# 计算抽取数量
num_test = int(len(train_poisoned) * TEST_RATIO)
num_normal = int(len(train_clean) * NORMAL_RATIO)

# 防止样本不够抽取
if num_test > len(train_poisoned): num_test = len(train_poisoned)
if num_normal > len(train_clean): num_normal = len(train_clean)

test_samples = random.sample(train_poisoned, num_test)
normal_samples = random.sample(train_clean, num_normal)

# 生成 test (从中毒样本抽取，保持中毒状态：有trigger，文本是目标类)
for s in test_samples:
    src = os.path.join(OUTPUT_DIR, "train", s["filename"])
    dst = os.path.join(OUTPUT_DIR, "test", s["filename"])
    shutil.copy(src, dst)

    rows.append({
        "raw": str([s["caption"]]),          # 目标类文本
        "origin": str([s["origin_caption"]]),# 原始类文本
        "sentids": str([sent_id]),
        "split": "test",
        "filename": s["filename"],
        "img_id": s["img_id"],
        "poisoned": 1
    })
    sent_id += 1

# 生成 normal (从干净样本抽取，修正文本为原始类别，图片不含trigger)
for s in normal_samples:
    src = os.path.join(OUTPUT_DIR, "train", s["filename"])
    dst = os.path.join(OUTPUT_DIR, "normal", s["filename"])
    shutil.copy(src, dst)

    # 核心修改点：Normal集的文本修改回原始类别
    # s["caption"] 在clean样本中等于 s["origin_caption"]
    normal_caption = [make_caption(TARGET_CLASS)]

    rows.append({
        "raw": str(normal_caption),        # 修改为原始文本
        "origin": str([s["origin_caption"]]),
        "sentids": str([sent_id]),
        "split": "normal",
        "filename": s["filename"],
        "img_id": s["img_id"],
        "poisoned": 0
    })
    sent_id += 1

# ==============================
# 保存 CSV
# ==============================
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print("Dataset generated:", OUTPUT_CSV)
print(f"Total Train: {len(train_samples)}, Test (Poisoned): {len(test_samples)}, Normal (Clean): {len(normal_samples)}")